In [1]:
# -------------------------------
# Step 1: Import required libraries
# -------------------------------

import numpy as np          # For numerical operations like arrays, max, argmax
import gym                  # To create the Taxi environment

# -------------------------------
# Step 2: Create the environment
# -------------------------------

# Create the Taxi environment (v3)
env = gym.make("Taxi-v3")

# Print environment details
print("Number of states:", env.observation_space.n)  # Total possible states (500)
print("Number of actions:", env.action_space.n)      # Total possible actions (6)

# -------------------------------
# Step 3: Initialize policy & value function
# -------------------------------

# Policy: for each state, choose a random action initially (0 to 5)
policy = np.random.choice(env.action_space.n, size=env.observation_space.n)

# Value function: start with zeros for all 500 states
value_table = np.zeros(env.observation_space.n)

# Discount factor (gamma) to prioritize immediate vs. future rewards
gamma = 0.9

# Convergence threshold for policy evaluation
theta = 1e-10


# -------------------------------
# Step 4: Policy Evaluation
# -------------------------------

def policy_evaluation(policy, value_table, env, gamma=0.9, theta=1e-10):
    """
    Evaluate the value function under the current policy.
    Iteratively update state values until convergence.
    """
    while True:
        delta = 0  # Tracks maximum change in value during an iteration
        # Loop through all 500 states
        for state in range(env.observation_space.n):
            v = 0
            action = policy[state]  # Action chosen by current policy
            # Look at possible transitions for this (state, action)
            for prob, next_state, reward, done in env.P[state][action]:
                # Bellman expectation equation
                # If done=True, there is no future value (value_table[next_state] is ignored)
                v += prob * (reward + (0 if done else gamma * value_table[next_state]))
            # Track max value change
            delta = max(delta, abs(value_table[state] - v))
            value_table[state] = v  # Update value of state
        # Stop if values converge (small change)
        if delta < theta:
            break
    return value_table


# -------------------------------
# Step 5: Policy Improvement
# -------------------------------

def policy_improvement(value_table, policy, env, gamma=0.9):
    """
    Improve the policy using the current value function.
    """
    policy_stable = True
    # Loop through all 500 states
    for state in range(env.observation_space.n):
        old_action = policy[state]  # Current action
        action_values = np.zeros(env.action_space.n)  # Store values of all 6 actions
        # Try all possible actions
        for action in range(env.action_space.n):
            for prob, next_state, reward, done in env.P[state][action]:
                # If done=True, there is no future value 
                action_values[action] += prob * (reward + (0 if done else gamma * value_table[next_state]))
        # Choose action with highest value
        policy[state] = np.argmax(action_values)
        # Check if policy changed
        if old_action != policy[state]:
            policy_stable = False
    return policy, policy_stable


# -------------------------------
# Step 6: Policy Iteration
# -------------------------------

def policy_iteration(env, gamma=0.9, theta=1e-10):
    """
    Run policy iteration algorithm.
    Returns optimal policy and value function.
    """
    # Initialize policy and value table
    policy = np.random.choice(env.action_space.n, size=env.observation_space.n)
    value_table = np.zeros(env.observation_space.n)

    while True:
        # Step 1: Evaluate current policy
        value_table = policy_evaluation(policy, value_table, env, gamma, theta)

        # Step 2: Improve policy based on new values
        policy, policy_stable = policy_improvement(value_table, policy, env, gamma)

        # Stop if policy no longer changes
        if policy_stable:
            return policy, value_table


# -------------------------------
# Step 7: Run Policy Iteration
# -------------------------------

optimal_policy, optimal_value = policy_iteration(env, gamma, theta)

print("\nOptimal Policy (best action for first 20 states):")
print(optimal_policy[:20])  # Printing first 20 states out ------of 500 for preview

print("\nOptimal Value Function (long-term value for first 20 states):")
print(optimal_value[:20])   # Cannot easily reshape 500 states to a simple grid like FrozenLake


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Number of states: 500
Number of actions: 6

Optimal Policy (best action for first 20 states):
[4 4 4 4 0 0 0 0 0 0 0 0 0 0 0 0 5 0 0 0]

Optimal Value Function (long-term value for first 20 states):
[17.          1.62261467  7.7147      2.9140163  -4.99684549  1.62261467
 -4.99684549 -3.13696226  1.62261467 -2.37440252  7.7147     -1.52711391
 -3.82326604 -2.37440252 -3.82326604  2.9140163  20.          2.9140163
  9.683       4.348907  ]
